# Comparación algortimos de Machine Learning 
Este notebook se enfoca en el desarrollo del codigo base para la creación, entrenamiento y evaluación de 3 modelos de machine learning para el data set `cardiovascular_disease.csv` con el objetivo de predeccir la variable caracteritica **cardio** la cual indica 0 para un paciente con enfermedad cardiovascular y un 1 por lo mismo.

In [15]:
# Importar librerías
from copy import deepcopy
import pandas as pd
import numpy as np
# Importar Decision Tree de scikit-learn
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
# Importar objeto para k-fold cross-validation con grid de hyperparámetros
from sklearn.model_selection import GridSearchCV
# Importar objeto para separación de datos en conjuntos de entrenamiento y evaluación
from sklearn.model_selection import train_test_split
# Importar objeto para escalamiento de datos
from sklearn.preprocessing import StandardScaler
# Importar metricas de evaluación
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve
# Importar objeto para imprimir matriz de confusión
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Importar objeto para imprimir reporte de clasificación
from sklearn.metrics import classification_report
from sklearn.model_selection import ParameterGrid
# Plots
import matplotlib.pyplot as plt 

In [17]:
#functions 

def train_M(model_ML, X_train, y_train, hiper_P={}, k=5, name="ML_train"):
    """
    Function that trains 3 different machine learning algorthms 
    Arguments:
        -model_ML: type of model to create, 3 suported options: DT for desicion trees 
            RF for random forest and SVM for suported vector machines 
        -X_train: set of all the variables to be used for training
        -y_train: labels of the variable of interest to be predicted in the X_train 
        -hiper_P: dictionary with the hiperparameter names as keys and a list for the values for 
            the grid for each hiperparameter 
        -name: name to be used in the res df
    """
    print(f"{'='*40}\nTraining {name} model\n{'='*40}")
    match model_ML:
        case "DT": 
            model_t=tree.DecisionTreeClassifier()
        case "RF": 
            model_t=RandomForestClassifier()
        case "SVM":
            model_t=SVC()
        case _: 
            return 0
    #hacer el grid de hiperparametros 
    model_f= GridSearchCV(model_t, hiper_P, verbose=2, scoring="f1_macro", cv=k)
    grid_df=pd.DataFrame(ParameterGrid(model_f.param_grid))
    print(grid_df)
    #entrenar el modelo 
    model_f.fit(X_train, y_train)
    res_df=pd.DataFrame({"metrics":["hiperparameters", f"F-1_test"],
        name:[model_f.best_params_, model_f.best_score_]
    })
    print(res_df)
    return [model_f, res_df, grid_df]

def test(model_t, X_test, y_test,type_m="macro", name="ML_test"):
    """
    function to test the model 
    Arguments: 
        -model_t: object with the model to be tested 
        -X_test: set of the variables needed to make the prediction
        -y_test: labels of the predicted data
        -type_m: tyope of the metrics to be used, macro or weighted
    """
    print(f"{'='*40}\nTesting {name} model{'='*40}")
    #make the prediction test 
    y_pred=model_t.predict(X_test)
    
    c_mat, axes = plt.subplots(1, 2, figsize=(12, 10))
    #make the confusion matrix
    confusion_matrix=confusion_matrix(y_test, y_pred)

    #absolute matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix,
                               display_labels=model_t.classes_)
    disp.plot(cmap=plt.cm.Blues)
    axes[0].set_title(f"Confusion matrix {name}")

    #normalized matrix 
    norm_confused_mat=confusion_matrix(y_test,y_pred, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=norm_confused_mat,
                               display_labels=model_t.classes_)
    disp.plot(cmap=plt.cm.Blues)
    axes[1].set_title(f"Confusion matrix normalized {name}")

    #Clasification report 
    report_class=classification_report(y_test, y_pred, output_dict=True)
    print(classification_report(y_test, y_pred))
    res_df=pd.DataFrame({"metrics":[f"F-1_score_{type_m}", f"precision_{type_m}", f"recall_{type_m}", f"AUROC_{type_m}", "F-1_score_weighted","precision_weighted","recall_weighted" ],
        name:[f1_score(y_test, y_pred), precision_score(y_test, y_pred),recall_score(y_test, y_pred),roc_auc_score(y_test, y_pred),report_class["weighted avg"]["f1-score"],report_class["weighted avg"]["precision"], report_class["weighted avg"]["recall"]]
    })
    return [res_df, c_mat, report_class]





## Descripción del data set 

El dataset esta formado por un total de 13 varibles, númericas y categoricas ambas en formato númerico, sin embargo una de ellas es redudante, puesto que se registra la edad en dias y años, por lo que para fines precticos solo se conservara en años.

In [3]:
data_set = pd.read_csv("/mnt/c/Users/vrahu/Documents/Ismadlsh/CCG/2026-S5/Data_Science/data/cardiovascular_disease.csv", index_col=0).rename_axis(None)
#eliminar la edad en días 
data_set.drop("age_days", axis=1, inplace=True)

display(data_set.head(6))
print("Dimensionalidad del data set:")
data_set.shape  

for col in data_set.columns: 
    non_defined_val=len(data_set[data_set[col]=="NA"].index) + len(data_set[data_set[col]==None].index)
    print(f"NA values in {col}: {non_defined_val}") 


var_interes="cardio"
print("Categorías únicas de clasificación")
data_set[var_interes].unique()

print(f"Num de intancias en la categoria {var_interes}:")
print(data_set.groupby(var_interes).size())

,age_year,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,50.391781,2,168,62.0,110,80,1,1,0,0,1,0
1,55.419178,1,156,85.0,140,90,3,1,0,0,1,1
2,51.663014,1,165,64.0,130,70,3,1,0,0,0,1
3,48.282192,2,169,82.0,150,100,1,1,0,0,1,1
4,47.873973,1,156,56.0,100,60,1,1,0,0,0,0
8,60.038356,1,151,67.0,120,80,2,2,0,0,0,0


Dimensionalidad del data set:
NA values in age_year: 0
NA values in gender: 0
NA values in height: 0
NA values in weight: 0
NA values in ap_hi: 0
NA values in ap_lo: 0
NA values in cholesterol: 0
NA values in gluc: 0
NA values in smoke: 0
NA values in alco: 0
NA values in active: 0
NA values in cardio: 0
Categorías únicas de clasificación
Num de intancias en la categoria cardio:
cardio
0    35021
1    34979
dtype: int64


## Preparación del data set 

### Separación de características y categorías de clase 

*   Guardar en X sólo características
*   Guardar en y sólo categorías de clase


In [4]:
X = data_set.loc[:,data_set.columns[data_set.columns != var_interes]]
print("Características")
print(X.head()) 


y = data_set.loc[:,var_interes]  # todos los renglones y sólo la columna 12
print("Categorias de clase")
print(y.head())

Características
    age_year  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  \
0  50.391781       2     168    62.0    110     80            1     1      0   
1  55.419178       1     156    85.0    140     90            3     1      0   
2  51.663014       1     165    64.0    130     70            3     1      0   
3  48.282192       2     169    82.0    150    100            1     1      0   
4  47.873973       1     156    56.0    100     60            1     1      0   

   alco  active  
0     0       1  
1     0       1  
2     0       0  
3     0       1  
4     0       0  
Categorias de clase
0    0
1    1
2    1
3    1
4    0
Name: cardio, dtype: int64


In [5]:
data_set.columns[data_set.columns != var_interes]

Index(['age_year', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo',
       'cholesterol', 'gluc', 'smoke', 'alco', 'active'],
      dtype='str')

### Escalamiento de los datos  

Se hizó la evluación de los modelos con escalamiento y sin escalamiento de datos para evaluar si este tipo de preparación influía en el mismo

In [6]:
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X) 

data_set_T={
    "scaled":{
        "X":X,
        "X_train":None,
        "y_train":None,
        "X_test":None,
        "y_test":None
    },
    "unscaled":{
        "X":X_scaled,
        "X_train":None,
        "y_train":None,
        "X_test":None,
        "y_test":None
    }
}

### Separación de sets datos

Ahora se hizo la separación de datos por cada uno de los dos tipos de datos con los que estamos trabajando (escalados y sin escalar), para ello se eligió una proporción 4-1 (80% datos de entrenamiento 20% de test) y un random state de 42 para segurar reproducibilidad en la función `train_test_split`

In [7]:

#fijar un random state para reproducibilida
random_state = 42

for data_t in data_set_T.keys():
    print(f"Separando datos de {data_t}")
    data_set_T[data_t]["X_train"], data_set_T[data_t]["X_test"], data_set_T[data_t]["y_train"], data_set_T[data_t]["y_test"] = train_test_split(data_set_T[data_t]["X"], y, test_size=0.2, random_state=random_state, stratify=y)

Separando datos de scaled
Separando datos de unscaled


## Modelos  

Se hizo el entrenamiento y evaluación de los 3 distintos algoritmos de aprendizaje automático con los dos distintos tipos de dataset.

In [18]:
res_df=pd.DataFrame
res_dic={
    "DT":{
        "HP":{'criterion':('gini', 'entropy'), 'class_weight':['balanced', None]}
    }, 
    "RF":{
        "HP":{'class_weight':['balanced', None], 'n_estimators':np.arange(50,550, 50), 'criterion': ['gini', 'entropy']}
    }, 
    "SVM":{
        "HP":{'kernel':('linear', 'rbf', 'poly'), 'C':np.arange(1,11,1)}
    }
} 

for ML_a in res_dic.keys(): 
    for data_t in data_set_T.keys():
        #make the training 
        res_dic[ML_a][data_t]=train_M(ML_a,data_set_T[data_t]["X_train"], data_set_T[data_t]["y_train"],res_dic[ML_a]["HP"], name=f"{ML_a}_{data_t}")


Training DT_scaled model
  class_weight criterion
0     balanced      gini
1     balanced   entropy
2          NaN      gini
3          NaN   entropy
Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ..............class_weight=balanced, criterion=gini; total time=   0.1s
[CV] END ..............class_weight=balanced, criterion=gini; total time=   0.2s
[CV] END ..............class_weight=balanced, criterion=gini; total time=   0.1s
[CV] END ..............class_weight=balanced, criterion=gini; total time=   0.2s
[CV] END ..............class_weight=balanced, criterion=gini; total time=   0.2s
[CV] END ...........class_weight=balanced, criterion=entropy; total time=   0.2s
[CV] END ...........class_weight=balanced, criterion=entropy; total time=   0.2s
[CV] END ...........class_weight=balanced, criterion=entropy; total time=   0.2s
[CV] END ...........class_weight=balanced, criterion=entropy; total time=   0.2s
[CV] END ...........class_weight=balanced, criterion=entropy;

KeyboardInterrupt: 